# Student Academic Performance Prediction

This notebook is an executable educational walkthrough of the UCI Student Performance dataset. It compares Linear Regression with and without earlier grades (`G1`, `G2`) and does not interpret observational correlations as causal effects.

## Dataset and objective

The target is the final mathematics grade, `G3`, on a 0–20 scale. The source is the UCI Student Performance repository. Earlier grades are available, so two experiments are reported to make their predictive contribution explicit.

In [ ]:
from pathlib import Path
import sys
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import train_test_split
from src.data_loader import download_dataset, load_dataset
from src.evaluate import regression_metrics, save_eda_figures
from src.preprocessing import build_model, split_features_target

csv_path = download_dataset(PROJECT_DIR / 'data' / 'raw')
data = load_dataset(csv_path)
data.shape, data.head()

## Inspection and exploratory analysis

In [ ]:
display(data.dtypes.to_frame('dtype'))
display(data.describe(include='all').T.head(10))
print('Missing values:', int(data.isna().sum().sum()))
print('Duplicate rows:', int(data.duplicated().sum()))
save_eda_figures(data, PROJECT_DIR / 'reports' / 'figures')
data.select_dtypes('number').corr()['G3'].sort_values(ascending=False).head(10)

The generated figures in `reports/figures/` show the target distribution, numeric correlations, and relationships with study time, absences, and failures. A correlation is an association in this sample, not proof that changing a feature causes a grade change.

## Shared split and leakage-safe model training

In [ ]:
train_idx, test_idx = train_test_split(range(len(data)), test_size=0.2, random_state=42)
train_data, test_data = data.iloc[list(train_idx)], data.iloc[list(test_idx)]
results = {}
predictions = {}
for name, include_grades in [('with_previous_grades', True), ('without_previous_grades', False)]:
    x_train, y_train = split_features_target(train_data, include_grades)
    x_test, y_test = split_features_target(test_data, include_grades)
    model = build_model(include_grades)
    model.fit(x_train, y_train)
    predictions[name] = model.predict(x_test)
    results[name] = regression_metrics(y_test, predictions[name])
baseline = DummyRegressor(strategy='mean').fit(pd.DataFrame(index=train_data.index), train_data['G3'])
predictions['dummy_baseline'] = baseline.predict(pd.DataFrame(index=test_data.index))
results['dummy_baseline'] = regression_metrics(test_data['G3'], predictions['dummy_baseline'])
pd.DataFrame(results).T

MAE is the average grade-point error; RMSE is also in grade points but penalizes large errors more; MSE is the squared version; and R² compares explained variance with predicting the training mean. The model including earlier grades is not a fair prospective predictor if those grades are unavailable at prediction time, which is why the second experiment matters.

In [ ]:
sample = data.drop(columns=['G3', 'G1', 'G2']).iloc[0].to_dict()
print('A sample feature record for the no-previous-grades model:')
print(sample)
print('Predicted grade:', float(predictions['without_previous_grades'][0]))

## Limitations and conclusion

The dataset is relatively small and observational, and its schools and context may not represent another population. Linear Regression assumes an additive linear relationship after encoding and can be influenced by outliers. The measured test metrics, rather than an assumed accuracy claim, should guide interpretation. The command-line application persists the same preprocessing and model together so inference uses the training feature schema.